In [1]:
import tensorflow as tf
import numpy as np

In [3]:
# representacion rectangular
def FFT_rect(images, shift=False, pad=False, to_gray=False, out_dim=None, expand_dims=True):
    outs = []
    for img in images:
        if to_gray: # pasa a escala de grises
            img = np.mean(img, axis=-1)

        aux = np.fft.fft2(img)
        if shift: # representacion centrada
            aux = np.fft.fftshift(aux)

        # representacion rectangular
        if pad:
            outs.append([np.pad(np.real(aux),2,'constant'), np.pad(np.imag(aux),2,'constant')])
        else:
            outs.append([np.real(aux), np.imag(aux)])
    outs = np.array(outs, dtype=np.float32)
    
    # redimension para construir representasion reducida
    if out_dim != None:
        r_mid = int(images.shape[1]/2)
        c_mid = int(images.shape[2]/2)
        dl = int(out_dim/2)
        outs = outs[:,:,r_mid-dl:r_mid+dl,c_mid-dl:c_mid+dl]

    # si se requiere expandir dimensiones
    if expand_dims:
        outs = np.expand_dims(outs, axis=-1)

    return outs


class LinearT(tf.keras.layers.Layer): # entra una imagen en escala de grises (sin canales)
    def __init__(self, out_dim): # out_dim es la dimension final m*m
        super().__init__()
        self.out_dim = out_dim

    def get_config(self):
        return {"out_dim": self.out_dim}

    def build(self, input_shape): # considera una dimension (batch, 2, n*n)
        self.rm = int(input_shape[2]/2)
        self.cm = int(input_shape[3]/2)

        # redimension por cuadrantes a flatt
        self.resh_flatt = tf.keras.layers.Reshape(target_shape=(2, 4, 1, self.rm*self.cm))

        # matrices de transformaciones lineales
        self.mt = self.add_weight(shape=(4, int(input_shape[2]/2)*int(input_shape[3]/2), int(self.out_dim/2)**2), initializer='random_normal', trainable=True)

        # redimension a patches
        self.resh_cs = tf.keras.layers.Reshape(target_shape=(2, 4, int(self.out_dim/2), int(self.out_dim/2), 1))

    def call(self, x): # (batch, cord, rows, cols)
        # cuadrantes
        q1 = x[:,:,:self.rm,:self.cm]
        q2 = x[:,:,:self.rm,self.cm:]
        q3 = x[:,:,self.rm:,:self.cm]
        q4 = x[:,:,self.rm:,self.cm:]
        qs = tf.stack([q1,q2,q3,q4], axis=2) # despues de la coordenada

        # representacion flatten
        flats = self.resh_flatt(qs)

        # transformaciones lineales
        mts = tf.matmul(flats, self.mt)
        chs = self.resh_cs(mts)

        # concatenacion de los cuadrantes
        up = tf.concat([chs[:,:,0], chs[:,:,1]], axis=3)
        down = tf.concat([chs[:,:,2], chs[:,:,3]], axis=3)
        ys = tf.concat([up, down], axis=2)

        return ys


class CReLU(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        return super().build(input_shape)

    def call(self, inputs):
        res = inputs[:,0]
        ims = inputs[:,1]

        return tf.stack([tf.nn.relu(res), tf.nn.relu(ims)], axis=1)


class modReLU(tf.keras.layers.Layer):
    def __init__(self, filters):
        super().__init__()
        self.filters = filters

    def build(self, input_shape):
        self.b = self.add_weight(shape=(self.filters,), name='bias',
                                  initializer=tf.keras.initializers.zeros(), trainable=True)

    def call(self, inputs):
        res = inputs[:,0]
        ims = inputs[:,1]

        coms = tf.complex(res,ims)
        mag = tf.math.abs(coms)
        pha = tf.math.angle(coms)

        rmag = tf.nn.relu(mag + self.b)

        return tf.stack([rmag*tf.math.cos(pha), rmag*tf.sin(pha)], axis=1)


class PhaseReLU(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        return super().build(input_shape)

    def call(self, inputs):
        res = inputs[:,0]
        ims = inputs[:,1]

        coms = tf.complex(res,ims)
        mag = tf.math.abs(coms)
        pha = tf.math.angle(coms)

        return tf.stack([mag*tf.math.cos(tf.nn.relu(pha)), mag*tf.sin(tf.nn.relu(pha))], axis=1)


class FReLU(tf.keras.layers.Layer):
    def __init__(self, filters):
        super().__init__()
        self.filters = filters

    def build(self, input_shape):
        self.b = self.add_weight(shape=(self.filters,), name='bias',
                                  initializer=tf.keras.initializers.zeros(), trainable=True)

    def call(self, inputs):
        res = inputs[:,0]
        ims = inputs[:,1]

        coms = tf.complex(res,ims)
        mag = tf.math.abs(coms)
        pha = tf.math.angle(coms)

        rmag = tf.nn.relu(mag - self.b**2)

        return tf.stack([rmag*tf.math.cos(pha), rmag*tf.sin(pha)], axis=1)


class RandomKernels(tf.keras.layers.Layer): # considera imagenes cuadradas de dimension (dim,dim,ch)
    def __init__(self, filters, act='relu'):
        super().__init__()
        self.filters = filters

        self.act = CReLU()

        if act == 'crelu':
            self.act = CReLU()
        elif act == 'modrelu':
            self.act = modReLU(filters=self.filters)
        elif act == 'prelu':
            self.act = PhaseReLU()
        elif act == 'frelu':
            self.act = FReLU(filters=self.filters)
        elif act == 'identity':
            self.act = tf.identity

    def get_config(self):
        return {"filters": self.filters, "act": self.act}


    def build(self, input_shape): # (batch, cord, row, cols, chan)
        # kernels aleatorios
        self.W_r = self.add_weight(
            shape=(input_shape[-3], input_shape[-2], input_shape[-1], self.filters),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05), trainable=False)
        self.W_i = self.add_weight(
            shape=(input_shape[-3], input_shape[-2], input_shape[-1], self.filters),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05), trainable=False)

        # escalas (iniciados como 1s)
        self.es_r = self.add_weight(shape=(input_shape[-1], self.filters), initializer="ones", trainable=True)
        self.es_i = self.add_weight(shape=(input_shape[-1], self.filters), initializer="ones", trainable=True)

    def call(self, x): # x es de shape (batch,coordenadas,r,c,chanels)
        x = tf.expand_dims(x, axis=-1) # agregamos una dimension al final para hacer broadcasting

        # calculo de la parte real
        r = x[:,0]*self.W_r - x[:,1]*self.W_i # aritmetica con broadcasting
        r = r*self.es_r
        r = tf.reduce_sum(r, axis=3) # suma sobre los canales (convolucion suma sobre canales)

        # calculo de la parte imaginaria
        i = x[:,0]*self.W_i + x[:,1]*self.W_r # aritmetica con broadcasting
        i = i*self.es_i
        i = tf.reduce_sum(i, axis=3) # suma sobre los canales (convolucion suma sobre canales)

        # stack de las coordenadas
        y = tf.stack([r, i], axis=1)

        return self.act(y) # se evalua con la funcion de activacion



class SpectralPooling(tf.keras.layers.Layer):
    def __init__(self, out_factor=0.5):
        super().__init__()
        self.of = out_factor

    def build(self, input_shape):
        self.x_min = int(input_shape[-3]*(1-self.of)/2) # indice minimo
        self.x_max = self.x_min + int(input_shape[-3]/2)

        self.y_min = int(input_shape[-2]*(1-self.of)/2) # indice minimo
        self.y_max = self.y_min + int(input_shape[-2]/2)
        

    def call(self, x):
        # itera sobre batch y coordenadas
        return x[:,:,self.x_min:self.x_max,self.y_min:self.y_max,:]


class Magnitude(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        pass

    def call (self, inputs):
        re = inputs[:,0,:,:,:]
        im = inputs[:,1,:,:,:]

        out = re*re + im*im

        return out

# 4x4 R-EF-CNN

In [5]:
# cargamos dataset sin normalizacion
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# obtenemos la representacion rectangular centrada
train_rect = FFT_rect(train_images, shift=True, expand_dims=True, out_dim=4)
test_rect = FFT_rect(test_images, shift=True, expand_dims=True, out_dim=4)

train_rect.shape, train_rect.dtype, test_rect.shape, test_rect.dtype

((60000, 2, 4, 4, 1), dtype('float32'), (10000, 2, 4, 4, 1), dtype('float32'))

In [6]:
# hyperparameters
epochs = 50
lr = 0.0005
momentum = 0.9
batch_size = 128

# architecture
inputs = tf.keras.Input((2,4,4,1))

# primera capa con 6 filtros
conv1 = RandomKernels(filters=6, act='crelu')(inputs)

# segunda capa con 16 filtros
conv2 = RandomKernels(filters=16, act='crelu')(conv1)

# tercera capa con 120 filtros
conv3 = RandomKernels(filters=120, act='crelu')(conv2)

# calculo de la magnitud
mag = Magnitude()(conv3)

# clasificacion
flat = tf.keras.layers.Flatten()(mag)
bn = tf.keras.layers.BatchNormalization()(flat)
dense = tf.keras.layers.Dense(84, activation='relu')(bn)
outputs = tf.keras.layers.Dense(10, activation='softmax')(dense)

# modelo
model = tf.keras.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["acc"]
)
model.fit(train_rect, train_labels, batch_size=batch_size, epochs=epochs,  validation_data=(test_rect, test_labels))

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 2, 4, 4, 1)]      0         
                                                                 
 random_kernels (RandomKerne  (None, 2, 4, 4, 6)       204       
 ls)                                                             
                                                                 
 random_kernels_1 (RandomKer  (None, 2, 4, 4, 16)      3264      
 nels)                                                           
                                                                 
 random_kernels_2 (RandomKer  (None, 2, 4, 4, 120)     65280     
 nels)                                                           
                                                                 
 magnitude (Magnitude)       (None, 4, 4, 120)         0         
                                                             

## 4x4 LT-EF-CNN

In [4]:
# cargamos dataset sin normalizacion
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# obtenemos la representacion rectangular centrada
train_rect = FFT_rect(train_images, shift=True, expand_dims=False)
test_rect = FFT_rect(test_images, shift=True, expand_dims=False)

train_rect.shape, train_rect.dtype, test_rect.shape, test_rect.dtype

((60000, 2, 28, 28), dtype('float32'), (10000, 2, 28, 28), dtype('float32'))

In [5]:
# hyperparameters
epochs = 50
lr = 0.0005
momentum = 0.9
batch_size = 128

# architecture
inputs = tf.keras.Input((2,28,28))

# capa lt
lt = LinearT(out_dim=4)(inputs)

# primera capa con 6 filtros
conv1 = RandomKernels(filters=6, act='crelu')(lt)

# segunda capa con 16 filtros
conv2 = RandomKernels(filters=16, act='crelu')(conv1)

# tercera capa con 120 filtros
conv3 = RandomKernels(filters=120, act='crelu')(conv2)

# calculo de la magnitud
mag = Magnitude()(conv3)

# clasificacion
flat = tf.keras.layers.Flatten()(mag)
bn = tf.keras.layers.BatchNormalization()(flat)
dense = tf.keras.layers.Dense(84, activation='relu')(bn)
outputs = tf.keras.layers.Dense(10, activation='softmax')(dense)

# modelo
model = tf.keras.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["acc"]
)
metrici = model.fit(train_rect, train_labels, batch_size=batch_size, epochs=epochs,  validation_data=(test_rect, test_labels))

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 2, 28, 28)]       0         
                                                                 
 linear_t (LinearT)          (None, 2, 4, 4, 1)        3136      
                                                                 
 random_kernels (RandomKerne  (None, 2, 4, 4, 6)       204       
 ls)                                                             
                                                                 
 random_kernels_1 (RandomKer  (None, 2, 4, 4, 16)      3264      
 nels)                                                           
                                                                 
 random_kernels_2 (RandomKer  (None, 2, 4, 4, 120)     65280     
 nels)                                                           
                                                             